# Practice 07 — Capstone Drill — SOLUTIONS

Worked answers. Try the learner notebook first!

In [ ]:
# Run me first. Imports + the practice toolkit (the grader and random task generators).
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')) if os.path.basename(os.getcwd()) != 'practice' else '.')
sys.path.insert(0, '.')
import torch
import torch.nn as nn
import practice_utils as pu
print('ready — torch', torch.__version__)

## Capstone drill — Maximize accuracy on a hard subset

A tough **5-class** draw with a higher bar (**86%**). Bring everything: a solid CNN (BatchNorm), regularization (dropout / weight decay), enough epochs, and early stopping. Iterate until you pass — then re-run the task cell and do it again with a new set of classes.

**Checklist:** ☐ CNN with ≥2 conv blocks ☐ BatchNorm ☐ dropout/weight-decay ☐ track val acc per epoch ☐ keep best weights ☐ ≥ 86% on the hidden test.

In [ ]:
task = pu.fashion_subset_task(k=5, train_per_class=1000, threshold=0.86)
task.describe()

In [ ]:
# ✅ reference solution
train_loader, test_loader = task.loaders()
k = task.n_classes
model = nn.Sequential(
    nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Dropout(0.3), nn.Linear(32*7*7,64), nn.ReLU(), nn.Linear(64,k))
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
lf = nn.CrossEntropyLoss()
import copy
best_acc, best_state, bad = 0.0, None, 0
for epoch in range(15):
    model.train()
    for xb, yb in train_loader:
        opt.zero_grad()
        lf(model(xb), yb).backward()
        opt.step()
    model.eval()
    correct = total = 0
    import torch as _t
    with _t.no_grad():
        for xb, yb in test_loader:
            correct += (model(xb).argmax(1)==yb).sum().item(); total += yb.size(0)
    acc = correct/total
    print('epoch', epoch+1, 'val acc', round(acc,4))
    if acc > best_acc:
        best_acc, best_state, bad = acc, copy.deepcopy(model.state_dict()), 0
    else:
        bad += 1
        if bad >= 5:
            print('early stop'); break
model.load_state_dict(best_state)   # restore BEST (not last) weights

In [ ]:
# check (run after your code)
task.grade(model)

🎓 Pass several hard draws and you're genuinely fluent. Bring this workflow to the capstone in `../07-capstone/`.